# Road Safety Open Data 2024

Student: Chenchen QIU  

Dataset: French *accidents corporels* (BAAC) 2024 from data.gouv.fr


| File | Grain | Content |
|------|-------|---------|
| caract | one accident | date, time, lighting, location, weather, collision |
| lieux | one road segment | road category, geometry, surface, speed limit |
| usagers | one person | role, severity, sex, birth year, safety equipment |
| vehicules | one vehicle | category, obstacle, impact point, maneuver |

We will follow a classic data lifecycle to conduct our analysis:  
Data Ingestion -> Data Profiling -> Data Transformation -> Data Modeling -> Data Enrichment -> Data Visualization.

## Data Ingestion

These raw files are already in the Medallion **Bronze** layer. We ingest them as-is.

In [ ]:
import numpy as np
import pandas as pd
from ydata_profiling import ProfileReport


caract = pd.read_csv("../data/caract-2024.csv", sep=";")
lieux = pd.read_csv("../data/lieux-2024.csv", sep=";")
usagers = pd.read_csv("../data/usagers-2024.csv", sep=";")
vehicules = pd.read_csv("../data/vehicules-2024.csv", sep=";")

print("caract   ", caract.shape)
print("lieux    ", lieux.shape)
print("usagers  ", usagers.shape)
print("vehicules", vehicules.shape)

/var/folders/xt/yvlbn6cd14n0_rc083ptj7n00000gn/T/ipykernel_68600/3888077576.py:3: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


caract    (54402, 15)
lieux     (70248, 18)
usagers   (125187, 16)
vehicules (92678, 11)


## Data Profiling

### Structure

We start by checking how pandas inferred each column's type.

In [2]:
print("=== caract ===")
print(caract.dtypes, "\n")
print("=== lieux ===")
print(lieux.dtypes, "\n")
print("=== usagers ===")
print(usagers.dtypes, "\n")
print("=== vehicules ===")
print(vehicules.dtypes)

=== caract ===
Num_Acc     int64
jour        int64
mois        int64
an          int64
hrmn       object
lum         int64
dep        object
com        object
agg         int64
int         int64
atm         int64
col         int64
adr        object
lat        object
long       object
dtype: object 

=== lieux ===
Num_Acc     int64
catr        int64
voie       object
v1          int64
v2         object
circ        int64
nbv        object
vosp        int64
prof        int64
pr         object
pr1        object
plan        int64
lartpc     object
larrout    object
surf        int64
infra       int64
situ        int64
vma         int64
dtype: object 

=== usagers ===
Num_Acc          int64
id_usager       object
id_vehicule     object
num_veh         object
place            int64
catu             int64
grav             int64
sexe             int64
an_nais        float64
trajet           int64
secu1            int64
secu2            int64
secu3            int64
locp             int64
actp   

Most identifiers and codes come through as `int64`, and `an_nais` as `float64` since it has blanks. A few columns stay `object` though, so we open them in Data Wrangler to browse the raw values and see why.

In `caract`, `adr`, `lat`, `long`, `dep` and `hrmn` turn out to be free text or use a decimal comma `,` that pandas doesn't parse on its own. `lieux` has the same issue for `nbv`, `pr`, `pr1`, `lartpc` and `larrout`, plus a space between thousands in `pr`/`pr1` (e.g. `3 591`, which looks like a normal space but isn't, so pandas can't read it as a number) and, in `nbv`, the `#VALEURMULTI` error values.

We'll fix this in Transformation: once the comma and thousands-space separators are stripped, `lat`, `long`, `nbv`, `pr`, `pr1`, `lartpc` and `larrout` parse cleanly to numbers (`nbv`'s `#VALEURMULTI` rows just become `NaN`, since there's nothing numeric to parse there), and `hrmn` turns into a proper time. `adr`, `dep` and `com` stay text, since they're an address and location codes rather than measurements.

### Automated alerts

We compute column-level alerts with `ydata-profiling` for each table.

In [3]:
for name, df in [
    ("caract", caract),
    ("lieux", lieux),
    ("usagers", usagers),
    ("vehicules", vehicules),
]:
    print(f"=== {name} ===")
    for a in (
        ProfileReport(df, title=name, minimal=True, progress_bar=False)
        .get_description()
        .alerts
    ):
        print(" ", a)
    print()

=== caract ===


100%|██████████| 15/15 [00:00<00:00, 91.12it/s]


  [an] has a constant value
  [adr] 2310 (4.2%) missing values
  [Num_Acc] has unique values

=== lieux ===


100%|██████████| 18/18 [00:00<00:00, 374.66it/s]


  [voie] 13331 (19.0%) missing values
  [v2] 64332 (91.6%) missing values
  [lartpc] 70215 (> 99.9%) missing values
  [v1] has 53866 (76.7%) zeros
  [vosp] has 57329 (81.6%) zeros
  [infra] has 58188 (82.8%) zeros

=== usagers ===


100%|██████████| 16/16 [00:00<00:00, 67.17it/s]


  [an_nais] 2579 (2.1%) missing values
  [id_usager] has unique values
  [trajet] has 35024 (28.0%) zeros
  [secu1] has 12216 (9.8%) zeros
  [secu2] has 47166 (37.7%) zeros
  [secu3] has 10516 (8.4%) zeros
  [locp] has 53151 (42.5%) zeros

=== vehicules ===


100%|██████████| 11/11 [00:00<00:00, 62.56it/s]

  [occutc] 91729 (99.0%) missing values
  [id_vehicule] has unique values
  [senc] has 4812 (5.2%) zeros
  [obs] has 78800 (85.0%) zeros
  [obsm] has 17572 (19.0%) zeros
  [choc] has 6083 (6.6%) zeros
  [manv] has 6801 (7.3%) zeros
  [motor] has 3888 (4.2%) zeros



### Missing values

Pandas already reads empty cells and `N/A` as `NaN`; BAAC additionally encodes "not specified" as the code `-1`. For completeness we count both as missing.

In [4]:
for name, df in [
    ("caract", caract),
    ("lieux", lieux),
    ("usagers", usagers),
    ("vehicules", vehicules),
]:
    # some -1 sentinels carry a leading space (" -1") in object columns, so strip before comparing
    stripped = df.astype(str).replace(r"^\s+|\s+$", "", regex=True)
    missing = df.isna() | (stripped == "-1")
    pct = (missing.mean() * 100).round(1)
    pct = pct[pct > 0].sort_values(ascending=False)
    print(f"=== {name} ===\n{pct.to_string() if len(pct) else '(none)'}\n")

=== caract ===
adr    4.2

=== lieux ===
lartpc     100.0
v2          91.6
larrout     69.2
pr1         39.1
pr          39.0
v1          23.2
voie        19.0
circ         6.2
nbv          5.9
vosp         5.5
vma          5.2
infra        1.2
prof         0.1
plan         0.1
surf         0.1
situ         0.1

=== usagers ===
etatp      91.8
secu3      90.4
locp       49.3
actp       49.3
secu2      43.0
an_nais     2.1
trajet      2.1
sexe        1.9
secu1       1.7

=== vehicules ===
occutc    99.0
motor      0.2
senc       0.1



`caract` and `vehicules` are close to complete, missing only `adr` (4.2%) and `occutc` (99%, but that's just non-transit accidents where the field doesn't apply). `lieux`'s geometry fields are the sparsest (`lartpc` 100%, `v2` 91.6%, `larrout` 69.2%, `pr`/`pr1` 39%, `v1` 23.2%, `voie` 19.0%), though its core fields (`catr`, `surf`, `situ`) stay under 0.2%. In `usagers`, the pedestrian-specific fields are only filled in when they apply (`etatp` 91.8%, `secu3` 90.4%, `locp`/`actp` 49.3%), but `grav` is complete, and `an_nais`/`sexe` are missing for only 2.1%/1.9% of rows.

So the large percentages above mostly reflect conditional applicability rather than data quality problems.

In Transformation, we'll map `-1` in `an_nais` and `sexe` to Unknown, and leave `lartpc`, `v2` and `occutc` out of modeling.

### Data integrity

We check referential integrity, grain and duplicates, coordinate and age ranges, and category domains.

In [5]:
accidents = set(caract["Num_Acc"])
print("lieux orphans:", len(set(lieux["Num_Acc"]) - accidents))
print("usagers orphans:", len(set(usagers["Num_Acc"]) - accidents))
print("vehicules orphans:", len(set(vehicules["Num_Acc"]) - accidents))
print("accidents without lieux:", len(accidents - set(lieux["Num_Acc"])))
print("accidents without usagers:", len(accidents - set(usagers["Num_Acc"])))
print("accidents without vehicules:", len(accidents - set(vehicules["Num_Acc"])))

lieux orphans: 0
usagers orphans: 0
vehicules orphans: 0
accidents without lieux: 0
accidents without usagers: 0
accidents without vehicules: 0


No orphan rows in either direction: every `lieux`, `usagers` and `vehicules` row maps back to a `caract` accident, and every accident shows up in all three files too. Referential integrity is complete.

In [6]:
multi = (lieux["Num_Acc"].value_counts() > 1).sum()
print(f"lieux multi-row accidents: {multi} ({multi / len(caract):.1%})")
print(
    "duplicate rows:",
    caract.duplicated().sum(),
    lieux.duplicated().sum(),
    usagers.duplicated().sum(),
    vehicules.duplicated().sum(),
)

lieux multi-row accidents: 15645 (28.8%)
duplicate rows: 0 2 0 0


15,645 accidents (28.8%) have more than one `lieux` row, which confirms it's road-segment grained rather than accident grained; these repeats aren't duplicates. There are 2 genuine duplicate rows though (accidents 202400012279 and 202400044389); the other three files have none.

In [7]:
print(f"usagers per accident (avg): {len(usagers) / len(caract):.2f}")
print(f"vehicules per accident (avg): {len(vehicules) / len(caract):.2f}")
print(
    "usagers with id_vehicule not in vehicules:",
    len(set(usagers["id_vehicule"]) - set(vehicules["id_vehicule"])),
)
print(
    f"usagers per vehicule (avg): {len(usagers) / vehicules['id_vehicule'].nunique():.2f}"
)

usagers per accident (avg): 2.30
vehicules per accident (avg): 1.70
usagers with id_vehicule not in vehicules: 0
usagers per vehicule (avg): 1.35


**Modeling implication:** since `caract` is 1-to-N with both `usagers` (2.30 persons per accident) and `vehicules` (1.70 vehicles per accident), and `usagers.id_vehicule` links cleanly N-to-1 to `vehicules` (no orphans, 1.35 persons per vehicule), `usagers` is the natural grain for the fact table, with `caract` and `vehicules` as dimensions. `lieux` is 1-to-N with `caract` too (28.8% multi-row), so unlike `vehicules` it can't sit at the fact grain either; it needs its own dimension.

In [8]:
# lat/long parsed locally from decimal-comma text; local check, not persisted (Profiling stage)
lat = pd.to_numeric(caract["lat"].str.replace(",", ".", regex=False), errors="coerce")
lon = pd.to_numeric(caract["long"].str.replace(",", ".", regex=False), errors="coerce")
metro = lat.between(41, 52) & lon.between(-5.5, 10)
overseas = caract["dep"].astype(str).str.startswith(("97", "98"))
print(
    f"lat {lat.min():.2f} to {lat.max():.2f}, long {lon.min():.2f} to {lon.max():.2f}"
)
print(
    f"outside metropolitan box: {(~metro).sum()} ({(~metro).mean():.1%}), overseas: {(~metro & overseas).sum()}"
)

lat -22.43 to 51.08, long -178.09 to 167.86
outside metropolitan box: 3347 (6.2%), overseas: 3344


3,347 points (6.2%) fall outside the metropolitan bounding box, but 3,344 of them are overseas departments, not bad data, and there are no `(0,0)` placeholders.

In [9]:
age = 2024 - usagers["an_nais"]
print(f"age {age.min():.0f} to {age.max():.0f}, negative ages: {(age < 0).sum()}")

age 0 to 110, negative ages: 0


Ages range from 0 to 110 with no negative values.

In [10]:
# categorical values outside their official domain (-1 = not specified, excluded)
print(
    "atm invalid:",
    sorted(set(caract["atm"].unique()) - set(range(1, 10)) - {-1}) or "none",
)
print(
    "col invalid:",
    sorted(set(caract["col"].unique()) - set(range(1, 8)) - {-1}) or "none",
)
print(
    "catr invalid:",
    sorted(set(lieux["catr"].unique()) - {1, 2, 3, 4, 5, 6, 7, 9} - {-1}) or "none",
)
print("grav invalid:", sorted(set(usagers["grav"].unique()) - {1, 2, 3, 4}) or "none")
print("catu invalid:", sorted(set(usagers["catu"].unique()) - {1, 2, 3}) or "none")
print("sexe invalid:", sorted(set(usagers["sexe"].unique()) - {1, 2} - {-1}) or "none")
print(
    "nbv anomaly (#VALEURMULTI):",
    (lieux["nbv"].astype(str).str.strip() == "#VALEURMULTI").sum(),
)

atm invalid: none
col invalid: none
catr invalid: none
grav invalid: none
catu invalid: none
sexe invalid: none
nbv anomaly (#VALEURMULTI): 50


All checked coded fields respect their official domains, except `lieux.nbv` which has 50 rows of the literal text `#VALEURMULTI` that are obviously error values.

### Quality summary

| Dimension | Finding |
|-----------|---------|
| Structure | Four files linked by `Num_Acc`, integrity complete |
| Grain | `lieux` road-segment grained (28.8% multi-row) — must aggregate before joining to `caract`, or accident counts inflate |
| Completeness | Mostly conditional applicability (e.g. pedestrian fields), not quality issues |
| Coordinates | 6.2% outside metropolitan box, nearly all (3,344/3,347) valid overseas departments — don't filter to metropolitan only |
| Domain validity | No negative ages; 50 `#VALEURMULTI` rows in `lieux.nbv` |
| Format artifacts | `lieux.pr`/`pr1`/`lartpc`/`larrout` stored as comma-decimal text; `pr`/`pr1` also have non-breaking-space thousands separators |
| Duplicates | 2 exact duplicate rows in `lieux`, none elsewhere |
| Sentinel values | `-1` (plain or leading-space) must map to Unknown, or it corrupts means like `vma` |

The data is otherwise clean, so Silver-layer work is standardization and deduplication, not a rebuild.

## Data Transformation (Silver layer)

We build the Silver tables from the issues found above: standardize formats (decimal comma, space thousands separator), replace the `-1` sentinel and the `nbv` `#VALEURMULTI` with `NaN`, and drop the 2 duplicate `lieux` rows.

In [11]:
caract_silver = caract.copy()
caract_silver["date"] = pd.to_datetime(
    dict(
        year=caract_silver["an"], month=caract_silver["mois"], day=caract_silver["jour"]
    )
)
caract_silver["time"] = pd.to_datetime(caract_silver["hrmn"], format="%H:%M").dt.time
caract_silver["lat"] = pd.to_numeric(
    caract_silver["lat"].str.replace(",", ".", regex=False), errors="coerce"
)
caract_silver["long"] = pd.to_numeric(
    caract_silver["long"].str.replace(",", ".", regex=False), errors="coerce"
)
caract_silver = caract_silver.replace(-1, np.nan)
caract_silver[["date", "time", "lat", "long"]].head(3)

,date,time,lat,long
0,2024-03-25,07:40:00,47.562770,6.758320
1,2024-03-20,15:05:00,47.021090,4.837550
2,2024-03-22,19:30:00,44.902384,2.496418


In [12]:
lieux_silver = lieux.drop_duplicates().copy()
for col in ["nbv", "lartpc", "larrout"]:
    lieux_silver[col] = pd.to_numeric(
        lieux_silver[col].str.strip().str.replace(",", ".", regex=False),
        errors="coerce",
    )
for col in ["pr", "pr1"]:
    lieux_silver[col] = pd.to_numeric(
        lieux_silver[col]
        .str.replace("\xa0", "", regex=False)
        .str.strip()
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )
lieux_silver = lieux_silver.replace(-1, np.nan)
print("lieux rows:", len(lieux), "->", len(lieux_silver))

lieux rows: 70248 -> 70246


In [13]:
usagers_silver = usagers.copy()
usagers_silver["actp"] = usagers_silver["actp"].str.strip()
usagers_silver = usagers_silver.replace([-1, "-1"], np.nan)

vehicules_silver = vehicules.replace(-1, np.nan)

We convert `lat`/`long` and the `lieux` measurement columns (`nbv`, `pr`, `pr1`, `lartpc`, `larrout`) from decimal-comma or non-breaking-space text to numbers, and build `date`/`time` in `caract_silver`. The `-1` sentinel, including its leading-space and letter-mixed `actp` forms, becomes `NaN` everywhere. That same numeric conversion also clears out `nbv`'s 50 `#VALEURMULTI` rows, since there's no number to parse there, while `pr`, `pr1`, `lartpc` and `larrout` convert just fine once their separators are stripped. Finally, we drop the 2 duplicate `lieux` rows, bringing it from 70,248 down to 70,246.

In [14]:
for name, df in [
    ("caract", caract_silver),
    ("lieux", lieux_silver),
    ("usagers", usagers_silver),
    ("vehicules", vehicules_silver),
]:
    m = (df.isna().mean() * 100).round(1)
    m = m[m > 0].sort_values(ascending=False)
    print(f"=== {name} ===\n{m.to_string() if len(m) else '(none)'}\n")
print("lieux_silver duplicate rows:", lieux_silver.duplicated().sum())

=== caract ===
adr    4.2

=== lieux ===
lartpc     100.0
v2          91.6
larrout     69.2
pr          39.0
pr1         39.0
v1          23.2
voie        19.0
circ         6.2
nbv          6.0
vosp         5.5
vma          5.2
infra        1.2
prof         0.1
plan         0.1
surf         0.1
situ         0.1

=== usagers ===
etatp      91.8
secu3      90.4
locp       49.3
actp       49.3
secu2      43.0
an_nais     2.1
trajet      2.1
sexe        1.9
secu1       1.7

=== vehicules ===
occutc    99.0
motor      0.2
senc       0.1

lieux_silver duplicate rows: 0


The missing percentages we get straight from `isna()` now match the workaround-based figures from profiling (`pr`/`pr1` at 39.0% each, `nbv` at 6.0%), and `lieux_silver` has zero duplicate rows left. The Silver tables are ready for Modeling.

We export the Silver tables to `data/silver/` for the Modeling stage.

In [15]:
caract_silver.to_csv("../data/silver/caract-2024.csv", index=False)
lieux_silver.to_csv("../data/silver/lieux-2024.csv", index=False)
usagers_silver.to_csv("../data/silver/usagers-2024.csv", index=False)
vehicules_silver.to_csv("../data/silver/vehicules-2024.csv", index=False)

## Data Modeling (Gold layer)

We build a star schema at the accident grain: `caract_silver` becomes `fact_accident`, and `usagers_silver`, `vehicules_silver` and `lieux_silver` become `dim_usager`, `dim_vehicule` and `dim_lieu`, each kept at its own natural grain and linked back to `fact_accident` via `Num_Acc`.

### Fact table and dimensions

`fact_accident` keeps its Silver grain (one row per `Num_Acc`). `dim_vehicule` also keeps its Silver grain (one row per `id_vehicule`) and keeps `Num_Acc` so it can link back to `fact_accident`. `lieux_silver` is road-segment grained (28.8% of accidents have multiple rows); we keep it as its own dimension at that grain, with a surrogate key.

In [16]:
fact_accident = caract_silver.drop(columns=["an", "jour", "mois", "hrmn"])
dim_vehicule = vehicules_silver.drop(columns=["num_veh"])
dim_lieu = lieux_silver.copy()
dim_lieu.insert(0, "id_lieu", range(len(dim_lieu)))

print("fact_accident:", fact_accident.shape)
print("dim_vehicule:", dim_vehicule.shape)
print("dim_lieu:", dim_lieu.shape)

fact_accident: (54402, 13)
dim_vehicule: (92678, 10)
dim_lieu: (70246, 19)


Since `dim_lieu` keeps its natural road-segment grain (70,246 rows, with `Num_Acc` repeating for 28.8% of accidents), we give it a surrogate `id_lieu` key. `fact_accident` drops `an`, `jour`, `mois` and `hrmn`, since `date` and `time` now cover that. `dim_vehicule` only drops `num_veh`, which `id_vehicule` supersedes, and keeps `Num_Acc` so it links back to `fact_accident`.

### Person dimension

`usagers_silver` stays at its natural person grain; we keep it as `dim_usager`, snowflaked off `fact_accident` via `Num_Acc` and off `dim_vehicule` via `id_vehicule`, and drop `num_veh`, redundant with `id_vehicule`.

In [17]:
dim_usager = usagers_silver.drop(columns=["num_veh"])
print("dim_usager:", dim_usager.shape)
print("id_usager unique:", dim_usager["id_usager"].is_unique)

dim_usager: (125187, 15)
id_usager unique: True


`dim_usager` ends up with 125,187 rows, one per `id_usager`, and that key is confirmed unique.

### Schema validation

We check that every dimension row resolves back to `fact_accident` (and `dim_usager` to `dim_vehicule`), and that each key is unique.

In [18]:
print(
    "dim_usager Num_Acc orphans (vs fact_accident):",
    len(set(dim_usager["Num_Acc"]) - set(fact_accident["Num_Acc"])),
)
print(
    "dim_vehicule Num_Acc orphans (vs fact_accident):",
    len(set(dim_vehicule["Num_Acc"]) - set(fact_accident["Num_Acc"])),
)
print(
    "dim_lieu Num_Acc orphans (vs fact_accident):",
    len(set(dim_lieu["Num_Acc"]) - set(fact_accident["Num_Acc"])),
)
print(
    "dim_usager id_vehicule orphans (vs dim_vehicule):",
    len(set(dim_usager["id_vehicule"]) - set(dim_vehicule["id_vehicule"])),
)
print("fact_accident Num_Acc unique:", fact_accident["Num_Acc"].is_unique)
print("dim_usager id_usager unique:", dim_usager["id_usager"].is_unique)
print("dim_vehicule id_vehicule unique:", dim_vehicule["id_vehicule"].is_unique)
print("dim_lieu id_lieu unique:", dim_lieu["id_lieu"].is_unique)

dim_usager Num_Acc orphans (vs fact_accident): 0
dim_vehicule Num_Acc orphans (vs fact_accident): 0
dim_lieu Num_Acc orphans (vs fact_accident): 0
dim_usager id_vehicule orphans (vs dim_vehicule): 0
fact_accident Num_Acc unique: True
dim_usager id_usager unique: True
dim_vehicule id_vehicule unique: True
dim_lieu id_lieu unique: True


Every `dim_usager`, `dim_vehicule` and `dim_lieu` row resolves back to `fact_accident`, and `dim_usager` resolves to `dim_vehicule` too. `Num_Acc`, `id_usager`, `id_vehicule` and `id_lieu` are each unique in their table, so the schema, with `fact_accident` at the center and `dim_usager`, `dim_vehicule` and `dim_lieu` snowflaked off it, holds together.

We export the Gold tables to `data/gold/`.

In [19]:
fact_accident.to_csv("../data/gold/fact_accident.csv", index=False)
dim_usager.to_csv("../data/gold/dim_usager.csv", index=False)
dim_vehicule.to_csv("../data/gold/dim_vehicule.csv", index=False)
dim_lieu.to_csv("../data/gold/dim_lieu.csv", index=False)